> 请点击获取[课程 PPT 内容](https://www.canva.cn/design/DAG3LntaIPg/NeNJCP8AM5XBnxItQhTasA/view?utm_content=DAG3LntaIPg&utm_campaign=designshare&utm_medium=link2&utm_source=uniquelinks&utlId=h7d4bc9e2d1)。


# 1. 环境配置

## 1.1 python 环境准备

In [ ]:
! pip install "langgraph-cli[inmem]" openai==2.11.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6 arxiv==2.3.1

## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

## 1.3 实践代码

为了能够顺利的演示内置中间件的使用详情，这里我们使用一段简单的智能体代码演示：

In [ ]:
from langchain_community.chat_models import ChatTongyi
import os
llm = ChatTongyi(api_key=os.environ.get("DASHSCOPE_API_KEY"), model="qwen-turbo")

from langchain_community.agent_toolkits.load_tools import load_tools
tools = load_tools(["arxiv"])

from langgraph.checkpoint.memory import InMemorySaver 
memory = InMemorySaver()

from langchain.agents import create_agent
agent = create_agent(model=llm, 
                     tools=tools, 
                     system_prompt="You are a helpful assistant", 
                     checkpointer=memory)

# 2. 智能体记忆

## 2.1 短期记忆管理

对于短期记忆而言，假如把所有的对话记录都进行保存，显然是不现实的。我们需要在使用的过程中定期的对其进行筛选（Filter），从而控制上下文的长度。

在 LangChain 中对短期记忆提供了以下几种处理方式：
- 自定义中间件：记忆剪裁（Memory Trimming） 和记忆删除（Memory Deletion）
- 内置中间件：记忆总结（SummarizationMiddleware）

### 2.1.1 记忆剪裁

在之前讲 RunnableWithMessageHistory 的时候就提到了如何利用 LangChain 提供的方法 trim_messages 进行记忆剪裁。

假如我们想要实现类似的效果，可以通过 before_model 的方式对 AgentState 内的 messages 信息进行调整:

In [ ]:
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from typing import Any

@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
  """只保留最新的三条信息"""
  messages = state["messages"]
  if len(messages) <= 3:
    return None
  first_msg = messages[0]
  recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
  new_messages = [first_msg] + recent_messages
  print("Trimmed messages:", new_messages)
  return {
    "messages": [
      RemoveMessage(id=REMOVE_ALL_MESSAGES),
      *new_messages]}

当然我们也可以将其载入到 agent 中，并多次调用进行测试：

In [ ]:
agent = create_agent(
  model=llm,
  tools=tools,
  system_prompt="You are a helpful assistant",
  middleware=[trim_messages],
  checkpointer=memory
)

config1 = {"configurable": {"thread_id": "1"}}

result1 = agent.invoke({"messages": [{"role": "user", "content": "请使用 arxiv 工具查询论文编号 1605.08386,"}]}, config=config1)
result2 = agent.invoke({"messages": [{"role": "user", "content": "没太懂，可以更详细的讲解一下吗"}]}, config=config1)
result3 = agent.invoke({"messages": [{"role": "user", "content": "还是不懂，你还记得我们查的是哪篇论文嘛？"}]}, config=config1)

print(result3)

### 2.1.2 记忆删除

在前面的记忆剪裁里，我们是通过 RemoveMessage(id=REMOVE_ALL_MESSAGES) 先把所有的信息删除，然后把我们需要的部分内容添加到列表中。

但其实可以灵活的运用 RemoveMessage(id=...) 的方法，根据要求定向的删除某些内容。比如说我们希望每次模型调用完后（after_model）删除掉最开始的两条信息，那我们就可以按下面的方式实现：

In [ ]:
from langchain.agents.middleware import after_model
@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
  """删除最前面两条的记忆"""
  messages = state["messages"]
  if len(messages) > 2:
    return {"messages": [RemoveMessage(id=m.id) for m in messages[:2]]}
  return None

然后将该中间件添加到 create_agent 中：

In [ ]:
agent = create_agent(
  model=llm,
  tools=tools,
  system_prompt="Please be concise and to the point.",
  middleware=[delete_old_messages],
  checkpointer=InMemorySaver(),
)

所以假如我们对其进行一次调用，此时就会把最初提问的 HumanMessage 和发出工具调用指令的 AIMessage 删除，从而让工具调用的结果 ToolMessage 作为第一条输出出来：

In [ ]:
config1 = {"configurable": {"thread_id": "2"}}
result1 = agent.invoke({"messages": [{"role": "user", "content": "请使用 arxiv 工具查询论文编号 1605.08386,"}]}, config=config1)
print(result1)

### 2.1.3 SummarizationMiddleware：对话历史摘要

这个中间件主要完成的任务是：
- 检测是否超过 token 阈值（比如 4000）
- 把历史消息进行摘要压缩
- 保留最新若干条消息（比如 20 条）

比如我们希望让最多 1000 token 的内容保存在上下文中，并且每次保留一轮的对话的话，那就需要下面这样进行设置（因为中文的 token 计算不太准确，所以我们这里以字符的方式计算总的 token 数量）：

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware
middleware = SummarizationMiddleware(
    model=llm,
    trigger=("tokens", 1000),
    keep=("messages", 1),
    token_counter=lambda messages: sum(len(m.content) for m in messages if hasattr(m, "content"))
)

将智能体与该内置中间件进行整合：

In [ ]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant",
    middleware=[middleware],
    checkpointer=InMemorySaver()
)

配置好后，我们可以连续让模型调用三次，然后打印一下最终的结果：

In [ ]:
config = {"configurable": {"thread_id": "3"}}

result1 = agent.invoke({"messages": [{"role": "user", "content": "请使用 arxiv 工具查询论文编号 1605.08386,"}]}, config=config)
result2 = agent.invoke({"messages": [{"role": "user", "content": "没太懂，可以更详细的讲解一下吗"}]}, config=config)
result3 = agent.invoke({"messages": [{"role": "user", "content": "还是不懂，你还记得我们查的是哪篇论文嘛？"}]}, config=config)

print(result3)

这样我们就成功实现了记忆的压缩了。但是除此之外我们还有一个问题是，这个记忆到底只是在传入给模型时候压缩了，还是实际存在 InMemorySaver() 也被改变了呢。

为了解决这个问题，我们可以将 agent 的记忆打印出来：

In [ ]:
snapshot = agent.get_state(config) # ✅ 关键
print(snapshot)           # 这里是一个 StateSnapshot

从结果上看，SummarizationMiddleware 并不是单纯的输入预处理， 它在执行过程中直接修改了智能体的内部状态（state）。所以可以看出这个中间件并不是仅在传入前将内容修改，而是修改后会同步更新给内部的短期记忆。我们可以通过部署在 LangSmith Studio 上进行更详尽的测试（Summarization_agent 文件夹）。

## 2.2 长期记忆

### 2.2.1 简介
不同于短期记忆解决的是模型在一次对话（thread）中如何记得刚才说过的话。长期记忆要解决的是模型如何在多次对话之间持续记住重要的事情，所以只有特定部分的内容会被记录，比如：
- 用户的兴趣（喜欢技术性回答）
- 历史事件（上次谈到了 LangGraph 的结构）
- 个性化偏好（希望回答简洁、英文风格）

长期记忆是通过一个名为 Store 的组件来实现的。我们可以把它理解成一个“键值数据库”，保存记忆文档（Memory Documents）。

每条记忆都会存在某个命名空间（namespace）下，拥有唯一的键（key），并且保存的内容是一个JSON 对象（value）。

我们可以使用各种方法不断扩展、更新和搜索，包括：
- .put() → 添加或更新记忆
- .get() → 获取指定记忆
- .search() → 搜索相关内容

### 2.2.2 添加并获取指定记忆
假如我们要向内写入一条长期记忆的话，可以通过：

In [ ]:
from langgraph.store.memory import InMemoryStore

user_id = "user_1"
application_context = "preferences"
namespace = (user_id, application_context)

store = InMemoryStore()

store.put(
    namespace,
    "a-memory",
    {
        "rules": [
            "User likes short, direct language",
            "User only speaks English & python",
        ],
        "my-key": "my-value",
    },
)


这里就是保存到 namespace 中，并且传入的 key 是 "a-memory", 对应的 value 是后面的字典内容（结构化的知识）。

假如我们希望从长期记忆里查找指定信息，可以通过 .get() 方法指定 namespace 和对应的 key 信息：

In [ ]:
item = store.get(namespace, "a-memory")
print(item)

此时就会将相关信息返回出来，但这种方法主要针对的是精确知道位置的情况下使用。

### 2.2.3 搜索相关记忆

但很多场景下，我们并不知道精确搜索的位置，此时我们就需要使用 RAG 的方式实现检索。

此时我们需要给 InMemoryStore 加入 Embedding 索引，让 .search() 支持向量语义检索（真实项目中 embed 需要换成你所用的 embedding 模型，dim 换成真实的维度）：

In [ ]:
from langgraph.store.memory import InMemoryStore

def embed(texts: list[str]) -> list[list[float]]:
    return [[1.0, 2.0] * len(texts)]

store = InMemoryStore(index={"embed": embed, "dims": 2}) 

我们就可以通过 .search 的方式在 namespace进行检索了。当然我们这里可以通过 filter 添加一些硬性条件（相当于 SQL 中的 where），然后 query 里就是对应查找的问题内容：

In [ ]:
user_id = "user_2"
application_context = "preferences"
namespace = (user_id, application_context)

store.put(
    namespace,
    "a-memory",
    {
        "rules": [
            "User likes short, direct language",
            "User only speaks English & python",
        ],
        "my-key": "my-value",
    },
)

# 搜索前需要先写入记忆
items = store.search( 
    namespace, filter={"my-key": "my-value"}, query="language preferences"
)
print(items)

### 2.2.4 实际应用

我们可以通过设计两个工具来进行长期记忆的保存和使用。比如我们首次传入在信息里传入相关信息，此时模型就会调用工具 save_user_info 保存相关的信息：
- user_id：abc123
- user_info：{"name": "Foo", "age": 25, "email": "foo@langchain.dev"}

In [ ]:
from typing import Any
from langgraph.store.memory import InMemoryStore
from langchain.tools import tool, ToolRuntime


# Access memory
@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """Look up user info."""
    store = runtime.store
    user_info = store.get(("users",), user_id)
    return str(user_info.value) if user_info else "Unknown user"

# Update memory
@tool
def save_user_info(user_id: str, user_info: dict[str, Any], runtime: ToolRuntime) -> str:
    """Save user info."""
    store = runtime.store
    store.put(("users",), user_id, user_info)
    return "Successfully saved user info."

store = InMemoryStore()
agent = create_agent(
    model=llm,
    tools=[get_user_info, save_user_info],
    store=store
)

# First session: save user info
result1 = agent.invoke({
    "messages": [{"role": "user", "content": "Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev"}]
})

print(result1)

当我们想要获取相关信息的话，此时模型会先基于 get_user_info 工具结合 user_id 信息找到对应的 content 信息，然后再基于这部分信息进行回复：

In [ ]:
# Second session: get user info
result2 = agent.invoke({
    "messages": [{"role": "user", "content": "Get user info for user with id 'abc123'"}]
})
print(result2)

# 3. 上下文工程
## 3.1 ContextEditingMiddleware
ContextEditingMiddleware 其核心任务是在调用模型之前，遍历消息列表，根据策略自动裁剪、修改、清除上下文消息，以达到控制 token 长度 / 避免冗余 的目的。

In [ ]:
from langchain.agents.middleware import ContextEditingMiddleware, ClearToolUsesEdit

middleware_context = ContextEditingMiddleware(
  edits=[
    ClearToolUsesEdit(
      trigger=200, # 超过 200 token清理
      keep=1,    # 保留最近 1 次
      placeholder="[旧结果已清理]",
      clear_tool_inputs=True
    )
  ]
)

agent = create_agent(
  model=llm,
  tools=tools,
  system_prompt="You are a helpful assistant",
  middleware=[middleware_context],
  checkpointer=memory
)

config2 = {"configurable": {"thread_id": "user_4"}}

result1 = agent.invoke({"messages": [{"role": "user", "content": "请使用 arxiv 工具查询论文编号 1605.08386，100字即可"}]}, config=config2)
result2 = agent.invoke({"messages": [{"role": "user", "content": "请使用 arxiv 工具查询论文编号 1706.03762，100字即可"}]}, config=config2)
result3 = agent.invoke({"messages": [{"role": "user", "content": "你能总结一下我们查过哪些论文吗？100字即可"}]}, config=config2)

print(result3)

又比如我们希望某些特定工具的信息不被清理，同时清理掉别的工具信息的话，我们可以加上：

In [ ]:
middleware_context = ContextEditingMiddleware(
  edits=[
    ClearToolUsesEdit(
      trigger=200, # 超过 200 token清理
      keep=1,    # 保留最近 1 次
      placeholder="[旧结果已清理]",
      clear_tool_inputs=True,
      exclude_tools=["arxiv"] # 不清理 arxiv 工具调用记录
    )
  ]
)

这个时候我们再运行上面的例子的话，arxiv 工具的记忆都会得以保留下来。

除了默认提供的 ClearToolUsesEdit 工具以外，其实我们还可以自己来进行打造类似的工具。本质上来说，任何我们定义的类，只要实现了一个和 LangChain 源码中 apply(messages, count_tokens=...) 方法，就可以当作一个编辑器（ContextEdit）来用。

比如，我们写一个「清除所有 AI 回复超过 500 tokens 的上下文」的编辑器：

In [ ]:
class TrimLongAIResponsesEdit:
    """清除过长 AI 回复的上下文编辑器"""
    def __init__(self, max_tokens_per_ai: int = 500):
        self.max_tokens_per_ai = max_tokens_per_ai

    def apply(self, messages: list, *, count_tokens):
        for i, msg in enumerate(messages):
            if isinstance(msg, AIMessage):
                tokens = count_tokens([msg])
                if tokens > self.max_tokens_per_ai:
                    # 替换过长内容
                    messages[i] = AIMessage(
                        content=f"[long response truncated: {tokens} tokens removed]",
                        id=msg.id,
                        additional_kwargs=getattr(msg, "additional_kwargs", {}))

agent = create_agent(
  model=llm,
  tools=tools,
  middleware=[
    ContextEditingMiddleware(
      edits=[
        TrimLongAIResponsesEdit(max_tokens_per_ai=500), 
      ]
    )
  ],
  checkpointer=memory
)